In [39]:
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path
import pandas as pd
import geopandas as gpd
import os
import ee

## Compare downloaded orbit files with tracks_year in the mgrs_df

In [32]:
data_dir = Path.home() / 'GEDI'
mgrs_file = data_dir/ 'mgrs_with_count_and_orbits.parquet'
mgrs_df = gpd.read_parquet(mgrs_file)
years = ['2019', '2020', '2021', '2022']
for year in years:
    print(f'Checking {year}')
    for i, row in mgrs_df.iterrows():
        if (data_dir/year/row['MGRS_UTM']).exists():
            norbits = len(os.listdir(data_dir/year/row['MGRS_UTM']))
            if (nwant := len(row[f'tracks_{year}'])) != norbits:
                print(f'{row["MGRS_UTM"]} has {norbits} orbits, not {nwant}')

Checking 2019
23J has 27 orbits, not 29
23K has 149 orbits, not 152
23L has 163 orbits, not 164
24K has 84 orbits, not 87
24L has 137 orbits, not 139
25M has 39 orbits, not 42
25S has 5 orbits, not 6
26Q has 12 orbits, not 13
26S has 16 orbits, not 20
27Q has 4 orbits, not 5
28M has 3 orbits, not 4
32M has 68 orbits, not 69
32N has 148 orbits, not 150
32P has 156 orbits, not 159
32S has 194 orbits, not 198
32T has 291 orbits, not 299
32U has 433 orbits, not 455
33H has 9 orbits, not 10
33J has 123 orbits, not 124
33K has 157 orbits, not 158
33L has 150 orbits, not 151
34H has 111 orbits, not 116
34M has 172 orbits, not 176
35H has 81 orbits, not 83
35J has 182 orbits, not 184
35K has 158 orbits, not 160
35L has 157 orbits, not 160
35M has 183 orbits, not 187
35U has 414 orbits, not 432
36K has 167 orbits, not 169
36L has 162 orbits, not 163
36M has 177 orbits, not 182
36U has 407 orbits, not 430
37K has 76 orbits, not 78
37M has 164 orbits, not 166
37U has 418 orbits, not 437
38K has 1

In [33]:
from download.utils import authenticate
authenticate()

Authenticating from keys/private-key.json
Authenticated Earth Engine successfully


In [49]:

zone='32P'
year = 2019
row = mgrs_df[mgrs_df['MGRS_UTM'] == zone].iloc[0]
geom = ee.Geometry.BBox(*row['geometry'].bounds).toGeoJSON()
last_coords = geom['coordinates'][0][0].copy()
geom['coordinates'][0].append(last_coords)
undownloaded_orbits = [o for o in row[f'tracks_{year}'] if not (data_dir/f'{year}/{zone}/{o.split("/")[-1]}.parquet').exists()]

In [50]:
filter = 'quality_flag==1 && degrade_flag==0 && region_class>0 && leaf_off_flag!=1'
sample_ratio = row['landmass'] * 0.7/row[f'count_{year}'] + 0.001
for o in undownloaded_orbits:
    fc = ee.FeatureCollection(o).filterBounds(geom).filter(filter)
    print(f'available, size: {fc.size().getInfo()}')
    fc = fc.randomColumn('random', seed=42).filter(ee.Filter.lte('random', sample_ratio))
    print(f'Downloading {o}, size: {fc.size().getInfo()}')

available, size: 3
available, size: 9
available, size: 1


## Check downloaded orbits and recorded orbits in mgrs_df

In [27]:
df = Path.home() /'GEDI'/ 'mgrs_with_count_and_orbits.parquet'
for i, row in df[['MGRS_UTM', 'tracks_2019']].iterrows():
    folder = Path.home() / 'GEDI' / '2019' / row['MGRS_UTM']
    if not os.path.exists(folder):
        print(f'{row["MGRS_UTM"]} does not exist')
        continue
    orbits = os.listdir(folder)
    for o in orbits:
        if f'LARSE/GEDI/GEDI02_A_002/{o.split(".")[0]}' not in row['tracks_2019']:
            print(f'{row["MGRS_UTM"]}, {o}')

26N does not exist
39M does not exist
40L does not exist
42P does not exist
43H does not exist
47L does not exist
53P does not exist
57P does not exist
58N does not exist
58U does not exist
59M does not exist
59P does not exist
59U does not exist
60L does not exist
01J does not exist
01M does not exist
01N does not exist
02M does not exist
03N does not exist
04M does not exist
07J does not exist
29G does not exist
02Q does not exist
03Q does not exist
03R does not exist
02U does not exist
03U does not exist
04U does not exist
05U does not exist
08U does not exist
42F does not exist
43F does not exist
31F does not exist
39G does not exist
59F does not exist
09K does not exist
10J does not exist
12P does not exist
13J does not exist
16N does not exist
17H does not exist
17J does not exist
23F does not exist
24F does not exist
25F does not exist


## Compare nwant and nsampled

In [83]:
data_dir = Path.home() / 'GEDI'
mgrs_file = Path.home() /'GEDI'/ 'mgrs_with_count_and_orbits.parquet'
mgrs_df = gpd.read_parquet(mgrs_file)
years = ['2019', '2020', '2021', '2022'] #, '2020', '2021', '2022'
nwant = mgrs_df['landmass']*0.7
for year in years:
    for file in (data_dir/year).glob('*.parquet'):
        num_rows = pq.read_metadata(file).num_rows
        mgrs_df.loc[mgrs_df['MGRS_UTM']==file.stem, f'sampled_{year}'] = num_rows
    mgrs_df = mgrs_df.fillna(0)
    mgrs_df = mgrs_df.astype({f'sampled_{year}': 'int64'}) #, 'sampled_2020': 'int64', 'sampled_2021': 'int64', 'sampled_2022': 'int64'
    # diff = sum((mgrs_df[f'sampled_{year}'] - nwant)>0)
    # print(diff)
mgrs_df.drop(columns=['tracks', 'geometry']).to_csv('~/GEDI/mgrs_sampled.csv', index=False)
mask = (mgrs_df['count_2019'] == 0) & (mgrs_df['count_2020'] == 0) & (mgrs_df['count_2021'] == 0) & (mgrs_df['count_2022'] == 0)
mgrs_df = mgrs_df[~mask]
mgrs_df.drop(columns=['tracks', 'geometry', 'tracks_2019', 'tracks_2020', 'tracks_2021', 'tracks_2022', 'tracks_2023']).to_csv('~/GEDI/mgrs_sampled.csv', index=False)